In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn_quantile import RandomForestQuantileRegressor
from tqdm.auto import tqdm
import CRPS.CRPS as pscore


import multiprocessing as mp
mp.set_start_method('spawn')

import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from quantile_regression import QuantileRegression


sys.path.append('../../../Evaluation/')
import conduct_evaluation
from normal_evaluation.quantile_regression_evaluation import *
from normal_evaluation.normal_evaluation import SampleOutcomes_Normal

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]


In [2]:
with open('./quantile_regression_models.pkl', 'rb') as f:
    quantile_regression_models = pickle.load(f)

In [3]:
with open('../../transformed_event_logs/BPIC_2017_all_test.pickle', 'rb') as f:
    test_data = pickle.load(f)

activity_count = [
 'W_Assess potential fraud__ate_abort',
 'W_Assess potential fraud__complete',
 'W_Assess potential fraud__resume',
 'W_Assess potential fraud__schedule',
 'W_Assess potential fraud__start',
 'W_Assess potential fraud__suspend',
 'W_Assess potential fraud__withdraw',
 'W_Call after offers__ate_abort',
 'W_Call after offers__complete',
 'W_Call after offers__resume',
 'W_Call after offers__schedule',
 'W_Call after offers__start',
 'W_Call after offers__suspend',
 'W_Call after offers__withdraw',
 'W_Call incomplete files__ate_abort',
 'W_Call incomplete files__complete',
 'W_Call incomplete files__resume',
 'W_Call incomplete files__schedule',
 'W_Call incomplete files__start',
 'W_Call incomplete files__suspend',
 'W_Complete application__ate_abort',
 'W_Complete application__complete',
 'W_Complete application__resume',
 'W_Complete application__schedule',
 'W_Complete application__start',
 'W_Complete application__suspend',
 'W_Handle leads__complete',
 'W_Handle leads__resume',
 'W_Handle leads__schedule',
 'W_Handle leads__start',
 'W_Handle leads__suspend',
 'W_Handle leads__withdraw',
 'W_Shortened completion __resume',
 'W_Shortened completion __schedule',
 'W_Shortened completion __start',
 'W_Shortened completion __suspend',
 'W_Validate application__ate_abort',
 'W_Validate application__complete',
 'W_Validate application__resume',
 'W_Validate application__schedule',
 'W_Validate application__start',
 'W_Validate application__suspend',
 ]

resource_count = [
'User_1',
 'User_10',
 'User_100',
 'User_101',
 'User_102',
 'User_103',
 'User_104',
 'User_105',
 'User_106',
 'User_107',
 'User_108',
 'User_109',
 'User_11',
 'User_110',
 'User_111',
 'User_112',
 'User_113',
 'User_114',
 'User_115',
 'User_116',
 'User_117',
 'User_118',
 'User_119',
 'User_12',
 'User_120',
 'User_121',
 'User_122',
 'User_123',
 'User_124',
 'User_125',
 'User_126',
 'User_127',
 'User_128',
 'User_129',
 'User_13',
 'User_130',
 'User_131',
 'User_132',
 'User_133',
 'User_134',
 'User_135',
 'User_136',
 'User_137',
 'User_138',
 'User_139',
 'User_14',
 'User_140',
 'User_141',
 'User_142',
 'User_143',
 'User_144',
 'User_145',
 'User_146',
 'User_147',
 'User_148',
 'User_149',
 'User_15',
 'User_16',
 'User_17',
 'User_18',
 'User_19',
 'User_2',
 'User_20',
 'User_21',
 'User_22',
 'User_23',
 'User_24',
 'User_25',
 'User_26',
 'User_27',
 'User_28',
 'User_29',
 'User_3',
 'User_30',
 'User_31',
 'User_32',
 'User_33',
 'User_34',
 'User_35',
 'User_36',
 'User_37',
 'User_38',
 'User_39',
 'User_4',
 'User_40',
 'User_41',
 'User_42',
 'User_43',
 'User_44',
 'User_45',
 'User_46',
 'User_47',
 'User_48',
 'User_49',
 'User_5',
 'User_50',
 'User_51',
 'User_52',
 'User_53',
 'User_54',
 'User_55',
 'User_56',
 'User_57',
 'User_58',
 'User_59',
 'User_6',
 'User_60',
 'User_61',
 'User_62',
 'User_63',
 'User_64',
 'User_65',
 'User_66',
 'User_67',
 'User_68',
 'User_69',
 'User_7',
 'User_70',
 'User_71',
 'User_72',
 'User_73',
 'User_74',
 'User_75',
 'User_76',
 'User_77',
 'User_78',
 'User_79',
 'User_8',
 'User_80',
 'User_81',
 'User_82',
 'User_83',
 'User_84',
 'User_85',
 'User_86',
 'User_87',
 'User_88',
 'User_89',
 'User_9',
 'User_90',
 'User_91',
 'User_92',
 'User_93',
 'User_94',
 'User_95',
 'User_96',
 'User_97',
 'User_98',
 'User_99',
]

ii1 = [
    'intercase_n_1__W_Assess potential fraud__ate_abort',
 'intercase_n_1__W_Assess potential fraud__complete',
 'intercase_n_1__W_Assess potential fraud__resume',
 'intercase_n_1__W_Assess potential fraud__schedule',
 'intercase_n_1__W_Assess potential fraud__start',
 'intercase_n_1__W_Assess potential fraud__suspend',
 'intercase_n_1__W_Assess potential fraud__withdraw',
 'intercase_n_1__W_Call after offers__ate_abort',
 'intercase_n_1__W_Call after offers__complete',
 'intercase_n_1__W_Call after offers__resume',
 'intercase_n_1__W_Call after offers__schedule',
 'intercase_n_1__W_Call after offers__start',
 'intercase_n_1__W_Call after offers__suspend',
 'intercase_n_1__W_Call after offers__withdraw',
 'intercase_n_1__W_Call incomplete files__ate_abort',
 'intercase_n_1__W_Call incomplete files__complete',
 'intercase_n_1__W_Call incomplete files__resume',
 'intercase_n_1__W_Call incomplete files__schedule',
 'intercase_n_1__W_Call incomplete files__start',
 'intercase_n_1__W_Call incomplete files__suspend',
 'intercase_n_1__W_Complete application__ate_abort',
 'intercase_n_1__W_Complete application__complete',
 'intercase_n_1__W_Complete application__resume',
 'intercase_n_1__W_Complete application__schedule',
 'intercase_n_1__W_Complete application__start',
 'intercase_n_1__W_Complete application__suspend',
 'intercase_n_1__W_Handle leads__complete',
 'intercase_n_1__W_Handle leads__resume',
 'intercase_n_1__W_Handle leads__schedule',
 'intercase_n_1__W_Handle leads__start',
 'intercase_n_1__W_Handle leads__suspend',
 'intercase_n_1__W_Handle leads__withdraw',
 'intercase_n_1__W_Shortened completion __resume',
 'intercase_n_1__W_Shortened completion __schedule',
 'intercase_n_1__W_Shortened completion __start',
 'intercase_n_1__W_Shortened completion __suspend',
 'intercase_n_1__W_Validate application__ate_abort',
 'intercase_n_1__W_Validate application__complete',
 'intercase_n_1__W_Validate application__resume',
 'intercase_n_1__W_Validate application__schedule',
 'intercase_n_1__W_Validate application__start',
 'intercase_n_1__W_Validate application__suspend',
]

ii3 = [
    'intercase_n_3__W_Assess potential fraud__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Assess potential fraud__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Assess potential fraud__complete_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Assess potential fraud__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__complete_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Assess potential fraud__suspend',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Assess potential fraud__resume',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Assess potential fraud__complete',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Assess potential fraud__suspend',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__withdraw_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__complete_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Assess potential fraud__resume',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Assess potential fraud__start_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__complete',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__suspend',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__withdraw_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Call after offers__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Call after offers__withdraw',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__start_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Call after offers__ate_abort',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__resume_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__schedule_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Call after offers__complete',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Shortened completion __schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Shortened completion __suspend',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__withdraw_W_Call after offers__schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__withdraw_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__complete_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Call after offers__ate_abort',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Shortened completion __schedule',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__start_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Call after offers__start_W_Shortened completion __suspend_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Call after offers__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Call after offers__start',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Call after offers__suspend_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__withdraw_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call after offers__withdraw_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__resume_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__complete_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Call incomplete files__complete',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Call incomplete files__suspend',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call incomplete files__ate_abort',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call incomplete files__resume',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Call incomplete files__complete',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Call incomplete files__suspend',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__complete_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Call incomplete files__ate_abort',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Call incomplete files__resume',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Call after offers__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Call incomplete files__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__complete',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__suspend',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Complete application__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Complete application__ate_abort_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Complete application__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Complete application__complete_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Complete application__resume_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__resume_W_Complete application__start_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__resume_W_Complete application__start_W_Complete application__suspend',
 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Complete application__ate_abort',
 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Complete application__resume_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Complete application__schedule',
 'intercase_n_3__W_Complete application__schedule_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Complete application__complete',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Complete application__suspend',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Shortened completion __schedule',
 'intercase_n_3__W_Complete application__schedule_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Complete application__start_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__start_W_Complete application__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__complete_W_Complete application__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Complete application__ate_abort',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Shortened completion __schedule',
 'intercase_n_3__W_Complete application__start_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Complete application__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Complete application__suspend_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__ate_abort_W_Complete application__schedule',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Complete application__start',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Complete application__suspend',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Shortened completion __schedule',
 'intercase_n_3__W_Complete application__suspend_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Handle leads__complete_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Handle leads__resume_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__complete_W_Handle leads__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__start_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__start_W_Handle leads__suspend',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__suspend_W_Handle leads__resume',
 'intercase_n_3__W_Handle leads__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Shortened completion __schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start_W_Handle leads__suspend',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__withdraw',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__withdraw_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Handle leads__start_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__start_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__start_W_Handle leads__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__start_W_Handle leads__suspend_W_Handle leads__resume',
 'intercase_n_3__W_Handle leads__suspend_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__suspend_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__complete',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__start',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__suspend',
 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Assess potential fraud__withdraw',
 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Handle leads__schedule',
 'intercase_n_3__W_Shortened completion __resume_W_Shortened completion __suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Shortened completion __resume_W_Validate application__ate_abort_W_Call incomplete files__schedule',
 'intercase_n_3__W_Shortened completion __resume_W_Validate application__resume_W_Validate application__suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__schedule',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Complete application__suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Shortened completion __suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Validate application__schedule',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__resume_W_Call after offers__suspend',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Shortened completion __start_W_Complete application__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Shortened completion __start_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Complete application__start',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Complete application__suspend',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Shortened completion __start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Shortened completion __suspend_W_Call after offers__resume_W_Call after offers__suspend',
 'intercase_n_3__W_Shortened completion __suspend_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __suspend_W_Complete application__start_W_Complete application__suspend',
 'intercase_n_3__W_Shortened completion __suspend_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Shortened completion __suspend_W_Shortened completion __resume_W_Shortened completion __suspend',
 'intercase_n_3__W_Validate application__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__ate_abort_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Validate application__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__complete_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Validate application__resume_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__resume_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__resume_W_Validate application__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__complete_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Validate application__complete',
 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Validate application__suspend',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Validate application__ate_abort',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Validate application__resume',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Validate application__complete',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Validate application__suspend',
 'intercase_n_3__W_Validate application__start_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__start_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Validate application__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Validate application__ate_abort',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Validate application__resume',
 'intercase_n_3__W_Validate application__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__suspend_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__ate_abort',
 'intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__resume',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__complete',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__start',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__suspend'
]

In [4]:
n_processes = 32
batch_size = 24
N = 1000

In [5]:
test_data

,Action_start,org:resource_start,concept:name,EventOrigin_start,EventID_start,lifecycle:transition_start,time:timestamp_start,case:LoanGoal_start,case:ApplicationType_start,case:concept:name,...,intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__ate_abort,intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__resume,intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Assess potential fraud__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Call incomplete files__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Validate application__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Assess potential fraud__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Call incomplete files__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__complete,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__start,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__suspend
6,Obtained,User_5,W_Call after offers__resume,Workflow,Workitem_1000019350,resume,2016-02-13 15:34:25.947000+00:00,Home improvement,New credit,Application_2110398373,...,0,0,0,0,0,0,0,0,0,9
18,Released,User_38,W_Call after offers__suspend,Workflow,Workitem_1000053568,suspend,2016-03-16 16:18:16.350000+00:00,Existing loan takeover,New credit,Application_1996091858,...,0,0,0,0,0,0,0,0,0,30
20,Obtained,User_29,W_Call incomplete files__start,Workflow,Workitem_1000065077,start,2016-11-29 15:46:43.081000+00:00,Existing loan takeover,New credit,Application_1196087089,...,0,0,0,0,0,0,0,0,0,11
24,Released,User_100,W_Validate application__suspend,Workflow,Workitem_1000078627,suspend,2016-04-04 10:28:20.976000+00:00,Car,New credit,Application_519522134,...,0,0,0,0,0,0,0,0,0,25
28,Obtained,User_52,W_Complete application__start,Workflow,Workitem_1000092342,start,2016-09-06 13:25:25.463000+00:00,Car,New credit,Application_159861644,...,0,0,0,0,0,0,0,0,0,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619775,Released,User_10,W_Complete application__suspend,Workflow,Workitem_999975609,suspend,2016-09-05 16:26:51.673000+00:00,Car,New credit,Application_1386592498,...,0,0,0,0,0,0,0,0,0,15
619776,Released,User_29,W_Validate application__suspend,Workflow,Workitem_999981856,suspend,2016-01-18 14:37:21.843000+00:00,Existing loan takeover,New credit,Application_947906225,...,0,0,0,0,0,0,0,0,0,15
619778,Obtained,User_28,W_Call incomplete files__resume,Workflow,Workitem_999990412,resume,2016-04-01 18:08:48.145000+00:00,Car,New credit,Application_883995052,...,0,0,0,0,0,0,0,0,0,12
619780,Created,User_118,W_Validate application__schedule,Workflow,Workitem_99999173,schedule,2016-04-13 07:49:14.822000+00:00,Not speficied,New credit,Application_52539020,...,0,0,0,0,0,0,0,0,0,40


In [6]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARS'], SampleOutcomes_QuantileRegression_ARS, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

  0%|                                                                                                 | 0/6068 [00:00<?, ?it/s]

  1%|▋                                                                                       | 50/6068 [00:01<02:17, 43.75it/s]

  2%|█▍                                                                                     | 100/6068 [00:01<01:18, 76.41it/s]

  2%|██▏                                                                                   | 150/6068 [00:01<00:56, 105.07it/s]

  3%|██▊                                                                                   | 200/6068 [00:02<00:49, 118.38it/s]

...

 95%|████████████████████████████████████  | 5763/6068 [00:23<00:00, 413.12it/s]

 96%|████████████████████████████████████▍ | 5813/6068 [00:24<00:01, 225.67it/s]

 97%|████████████████████████████████████▉ | 5905/6068 [00:24<00:00, 278.29it/s]

 99%|█████████████████████████████████████▋| 6025/6068 [00:24<00:00, 348.61it/s]

100%|██████████████████████████████████████| 6068/6068 [00:24<00:00, 248.16it/s]

In [7]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-6.041137833916020830245688576')

In [8]:
np.mean(get_pscores(likelihoods_A))

np.float64(1008307.318868637)

In [9]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSAC'], SampleOutcomes_QuantileRegression_ARSAC, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

  0%|                                                  | 0/6068 [00:00<?, ?it/s]

  1%|▎                                        | 50/6068 [00:00<01:59, 50.55it/s]

  2%|▋                                       | 100/6068 [00:01<01:05, 91.77it/s]

  2%|▉                                      | 150/6068 [00:01<00:47, 124.22it/s]

  3%|█▎                                     | 200/6068 [00:01<00:39, 148.98it/s]

...

 95%|████████████████████████████████████  | 5761/6068 [00:23<00:00, 335.23it/s]

 96%|████████████████████████████████████▍ | 5811/6068 [00:23<00:00, 312.60it/s]

 97%|████████████████████████████████████▉ | 5905/6068 [00:24<00:00, 264.46it/s]

 99%|█████████████████████████████████████▌| 6001/6068 [00:24<00:00, 330.99it/s]

100%|██████████████████████████████████████| 6068/6068 [00:24<00:00, 249.12it/s]

In [10]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-6.675715801792923844242381931')

In [11]:
np.mean(get_pscores(likelihoods_A))

np.float64(1643629.6262623873)

In [12]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSRC'], SampleOutcomes_QuantileRegression_ARSRC, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

  0%|                                                  | 0/6068 [00:00<?, ?it/s]

  1%|▎                                        | 50/6068 [00:00<01:54, 52.67it/s]

  2%|▋                                       | 100/6068 [00:01<01:02, 95.33it/s]

  2%|▉                                      | 150/6068 [00:01<00:45, 128.84it/s]

  3%|█▎                                     | 200/6068 [00:01<00:37, 155.46it/s]

...

 94%|███████████████████████████████████▊  | 5713/6068 [00:23<00:01, 294.01it/s]

 97%|████████████████████████████████████▋ | 5857/6068 [00:24<00:00, 315.40it/s]

 98%|█████████████████████████████████████▏| 5929/6068 [00:24<00:00, 331.89it/s]

 99%|█████████████████████████████████████▋| 6025/6068 [00:24<00:00, 399.23it/s]

100%|██████████████████████████████████████| 6068/6068 [00:24<00:00, 248.80it/s]

In [13]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-6.071411773671639243241662429')

In [14]:
np.mean(get_pscores(likelihoods_A))

np.float64(1316548.895443666)

In [15]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSACRC'], SampleOutcomes_QuantileRegression_ARSACRC, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

  0%|                                                  | 0/6068 [00:00<?, ?it/s]

  1%|▎                                        | 50/6068 [00:01<02:08, 46.73it/s]

  2%|▋                                       | 100/6068 [00:01<01:10, 84.37it/s]

  2%|▉                                      | 150/6068 [00:01<00:50, 117.75it/s]

  3%|█▎                                     | 200/6068 [00:01<00:40, 146.07it/s]

...

 93%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉                | 5665/6068 [00:23<00:01, 232.03it/s]

 96%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋         | 5833/6068 [00:24<00:00, 327.27it/s]

 98%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍    | 5953/6068 [00:24<00:00, 423.24it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6068/6068 [00:24<00:00, 250.38it/s]

Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps


In [16]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-Infinity')

In [17]:
np.mean(get_pscores(likelihoods_A))

np.float64(1729883.4066740386)

In [18]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSD'], SampleOutcomes_QuantileRegression_ARSD, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

  0%|                                                                                                                                                                                                                                                             | 0/6068 [00:00<?, ?it/s]

  1%|██                                                                                                                                                                                                                                                  | 50/6068 [00:01<02:09, 46.58it/s]

  2%|████                                                                                                                                                                                                                                               | 100/6068 [00:01<01:11, 84.05it/s]

  2%|█████▉                                                                                                                                                                                                                                            | 150/6068 [00:01<00:52, 113.31it/s]

  3%|███████▉                                                                                                                                                                                                                                          | 200/6068 [00:01<00:43, 135.47it/s]

...

 94%|██████████████████████████████████████████████████████████████████████████████▊     | 5689/6068 [00:43<00:01, 205.38it/s]

 95%|███████████████████████████████████████████████████████████████████████████████▍    | 5739/6068 [00:44<00:01, 205.40it/s]

 97%|█████████████████████████████████████████████████████████████████████████████████   | 5857/6068 [00:44<00:00, 215.44it/s]

 98%|██████████████████████████████████████████████████████████████████████████████████  | 5929/6068 [00:44<00:00, 203.67it/s]

100%|████████████████████████████████████████████████████████████████████████████████████| 6068/6068 [00:44<00:00, 134.96it/s]

In [19]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-6.028094766969217331232185208')

In [20]:
np.mean(get_pscores(likelihoods_A))

np.float64(1046865.5735045165)

In [21]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRC'], SampleOutcomes_QuantileRegression_ARSDACRC, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

  0%|                                                                                                | 0/6068 [00:00<?, ?it/s]

  1%|▋                                                                                      | 50/6068 [00:01<02:22, 42.25it/s]

  2%|█▍                                                                                    | 100/6068 [00:01<01:21, 73.53it/s]

  2%|██▏                                                                                   | 150/6068 [00:01<01:01, 96.73it/s]

  3%|██▊                                                                                  | 200/6068 [00:02<00:51, 113.77it/s]

...

 96%|███████████████████████████████████████████████████████████████▍  | 5833/6068 [00:23<00:00, 337.01it/s]

 97%|███████████████████████████████████████████████████████████████▉  | 5883/6068 [00:24<00:00, 331.22it/s]

 99%|█████████████████████████████████████████████████████████████████ | 5977/6068 [00:24<00:00, 418.78it/s]

 99%|█████████████████████████████████████████████████████████████████▌| 6027/6068 [00:24<00:00, 429.69it/s]

100%|██████████████████████████████████████████████████████████████████| 6068/6068 [00:24<00:00, 249.06it/s]

In [22]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-7.035835662785683947288057503')

In [23]:
np.mean(get_pscores(likelihoods_A))

np.float64(1830233.2733803245)

In [24]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRCII1'],
                                                   SampleOutcomes_QuantileRegression_ARSDACRCII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'inter_instance_column_names' : ii1
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)


  0%|                                                                              | 0/6068 [00:00<?, ?it/s]

  1%|▌                                                                    | 50/6068 [00:01<02:09, 46.53it/s]

  2%|█                                                                   | 100/6068 [00:01<01:11, 83.66it/s]

  2%|█▋                                                                 | 150/6068 [00:01<00:52, 112.70it/s]

  3%|██▏                                                                | 200/6068 [00:01<00:43, 134.99it/s]

...

 95%|██████████████████████████████████████████████████████████████▉   | 5785/6068 [00:23<00:00, 419.72it/s]

 97%|███████████████████████████████████████████████████████████████▋  | 5857/6068 [00:23<00:00, 395.45it/s]

 97%|████████████████████████████████████████████████████████████████▏ | 5907/6068 [00:24<00:00, 267.34it/s]

 99%|█████████████████████████████████████████████████████████████████▎| 6001/6068 [00:24<00:00, 339.61it/s]

100%|██████████████████████████████████████████████████████████████████| 6068/6068 [00:24<00:00, 248.62it/s]

In [25]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-6.901494676035567707947543929')

In [26]:
np.mean(get_pscores(likelihoods_A))

np.float64(1811523.5491065898)

In [27]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRCII3'],
                                                   SampleOutcomes_QuantileRegression_ARSDACRCII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'inter_instance_column_names' : ii3
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

  0%|                                                                              | 0/6068 [00:00<?, ?it/s]

  1%|▌                                                                    | 50/6068 [00:00<01:54, 52.38it/s]

  2%|█                                                                   | 100/6068 [00:01<01:02, 95.15it/s]

  2%|█▋                                                                 | 150/6068 [00:01<00:46, 128.23it/s]

  3%|██▏                                                                | 200/6068 [00:01<00:37, 154.61it/s]

...

 93%|█████████████████████████████████████████████████    | 5617/6068 [00:23<00:01, 244.69it/s]

 97%|███████████████████████████████████████████████████▏ | 5857/6068 [00:24<00:00, 351.27it/s]

 98%|███████████████████████████████████████████████████▉ | 5953/6068 [00:24<00:00, 422.69it/s]

 99%|████████████████████████████████████████████████████▍| 6003/6068 [00:24<00:00, 327.91it/s]

100%|█████████████████████████████████████████████████████| 6068/6068 [00:24<00:00, 248.20it/s]

In [28]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-6.870004543705421007280387925')

In [29]:
np.mean(get_pscores(likelihoods_A))

np.float64(1781716.1126756158)

In [30]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDII1'],
                                                   SampleOutcomes_QuantileRegression_ARSDII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'inter_instance_column_names' : ii1
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

  0%|                                                                 | 0/6068 [00:00<?, ?it/s]

  1%|▍                                                       | 50/6068 [00:01<02:09, 46.58it/s]

  2%|▉                                                      | 100/6068 [00:01<01:11, 83.92it/s]

  2%|█▎                                                    | 150/6068 [00:01<00:52, 112.90it/s]

  3%|█▊                                                    | 200/6068 [00:01<00:43, 135.02it/s]

...

 94%|███████████████████████████████████████████████████████████████████████████████████████▌     | 5713/6068 [00:23<00:01, 309.25it/s]

 96%|█████████████████████████████████████████████████████████████████████████████████████████    | 5809/6068 [00:24<00:00, 365.23it/s]

 97%|██████████████████████████████████████████████████████████████████████████████████████████▏  | 5881/6068 [00:24<00:00, 416.66it/s]

 99%|███████████████████████████████████████████████████████████████████████████████████████████▉ | 6001/6068 [00:24<00:00, 410.57it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 6068/6068 [00:24<00:00, 248.33it/s]

In [31]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-8.452255740190578936243376663')

In [32]:
np.mean(get_pscores(likelihoods_A))

np.float64(1037674.5222693885)

In [33]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDII3'],
                                                   SampleOutcomes_QuantileRegression_ARSDII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'inter_instance_column_names' : ii3
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

  0%|                                                                                                         | 0/6068 [00:00<?, ?it/s]

  1%|▊                                                                                               | 50/6068 [00:01<02:09, 46.36it/s]

  2%|█▌                                                                                             | 100/6068 [00:01<01:11, 83.50it/s]

  2%|██▎                                                                                           | 150/6068 [00:01<00:52, 112.43it/s]

  3%|███                                                                                           | 200/6068 [00:01<00:43, 134.62it/s]

...

 92%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉                 | 5583/6068 [00:48<00:03, 157.23it/s]

 94%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋             | 5689/6068 [00:48<00:02, 152.23it/s]

 95%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏          | 5761/6068 [00:49<00:02, 138.98it/s]

 97%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌       | 5857/6068 [00:49<00:01, 181.77it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6068/6068 [00:49<00:00, 121.72it/s]

In [34]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-217.7754561840178173395621284')

In [35]:
np.mean(get_pscores(likelihoods_A))

np.float64(1013939.1709911189)